# Jailbreak prompt classification

**Task** — train a binary classifier that separates *jailbreak* prompts from *benign* prompts and
emits a jailbreak likelihood in `[0, 1]` for every prompt in `test.csv`.

The exported `submission.csv` uses the schema required by the issue:

| Column | Meaning |
| --- | --- |
| `Id` | the prompt id, copied verbatim from `test.csv` |
| `TARGET` | jailbreak likelihood in `[0, 1]`; `TARGET >= 0.5` -> **jailbreak**, `TARGET < 0.5` -> **benign** |

### Project layout

The notebook is intentionally thin: every piece of logic lives in a numbered Python module that the
cells below import and call. The modules are listed **in the order they are used**.

| # | Module | Responsibility |
| --- | --- | --- |
| 1 | `1_data_loader.py` | Load `train.csv` / `test.csv`, validate schema, assert no train/test leakage |
| 2 | `2_preprocessor.py` | Unicode/whitespace normalisation, the **word-ID** encoder, style features |
| 3 | `3_feature_builder.py` | TF-IDF word / character feature blocks and the full feature union |
| 4 | `4_model_trainer.py` | Near-duplicate grouping, base models, grouped CV, stacking |
| 5 | `5_evaluator.py` | ROC-AUC and supporting metrics, threshold sweep, bootstrap CI |
| 6 | `6_submission_exporter.py` | Build / validate / write `submission.csv` |

### Approach

1. **Safeguards first.** The train and test sets are checked for label conflicts, duplicate prompts and
   normalised-text overlap *before* any model is fitted. Near-duplicate prompts (template twins) are
   clustered and pinned to a single cross-validation fold, so the reported score cannot be inflated by
   leakage.
2. **Two complementary views of the prompt.** Word n-gram TF-IDF captures the vocabulary a jailbreak
   uses; character n-gram TF-IDF captures the shape of the text and survives typos, casing tricks and
   zero-width characters. The literature on prompt-injection detection consistently finds the character
   view generalises better against obfuscation.
3. **Stacked ensemble.** Four base models (word/character x logistic-regression/linear-SVM) produce
   out-of-fold probabilities; a logistic meta-learner combines them.
4. **Every number is out-of-fold.** No reported metric is computed on a row the model was trained on.

## Cell 1 — configuration and imports

In [ ]:
import importlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- configuration -----------------------------------------------------
BASE_DIR = Path.cwd()
TRAIN_CSV = BASE_DIR / "train.csv"
TEST_CSV = BASE_DIR / "test.csv"
SAMPLE_SUBMISSION_CSV = BASE_DIR / "sample_submission.csv"
OUTPUT_CSV = BASE_DIR / "submission.csv"

N_SPLITS = 5          # grouped cross-validation folds
RANDOM_STATE = 42
DECISION_THRESHOLD = 0.5   # issue rule: TARGET >= 0.5 -> jailbreak

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# The project modules are named in list order; import them by that name.
data_loader = importlib.import_module("1_data_loader")
preprocessor = importlib.import_module("2_preprocessor")
feature_builder = importlib.import_module("3_feature_builder")
model_trainer = importlib.import_module("4_model_trainer")
evaluator = importlib.import_module("5_evaluator")
submission_exporter = importlib.import_module("6_submission_exporter")

MODULES = [
    "1_data_loader",
    "2_preprocessor",
    "3_feature_builder",
    "4_model_trainer",
    "5_evaluator",
    "6_submission_exporter",
]

print(f"Python  : {sys.version.split()[0]}")
print(f"numpy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")
import sklearn
print(f"sklearn : {sklearn.__version__}")
print(f"working dir: {BASE_DIR}")
print("modules :", ", ".join(MODULES))

## Cell 2 — load the data and run the safeguards

In [ ]:
train_df, test_df = data_loader.load_data(str(TRAIN_CSV), str(TEST_CSV))
sample_submission = data_loader.get_submission_schema(str(SAMPLE_SUBMISSION_CSV))

# Raises on: label drift, duplicate ids, id overlap, or train/test text leakage.
data_loader.validate_data(train_df, test_df)

print(f"train rows      : {len(train_df)}")
print(f"test rows       : {len(test_df)}")
print(f"sample sub rows : {len(sample_submission)}")
print()
print(train_df['label'].value_counts().to_string())
print()
print(f"train Id range  : {train_df['Id'].min()} .. {train_df['Id'].max()}")
print(f"test  Id range  : {test_df['Id'].min()} .. {test_df['Id'].max()}")
print()
train_df.head(3).to_string()

## Cell 3 — normalise the prompts

NFKC folding, zero-width character removal, lowercasing and whitespace collapsing. This runs inside
every model's pipeline as well, so the training and inference paths can never diverge.

In [ ]:
train_texts = train_df['text'].astype(str).tolist()
test_texts = test_df['text'].astype(str).tolist()

train_norm = preprocessor.normalize_series(train_texts)
test_norm = preprocessor.normalize_series(test_texts)

print(f"train chars: {sum(len(t) for t in train_texts):,} -> {sum(len(t) for t in train_norm):,} after normalisation")
print(f"test  chars: {sum(len(t) for t in test_texts):,} -> {sum(len(t) for t in test_norm):,} after normalisation")
print()
sample_idx = 0
print("--- raw ---")
print(repr(train_texts[sample_idx][:220]))
print("--- normalised ---")
print(repr(train_norm[sample_idx][:220]))

## Cell 4 — word-ID demonstration

The issue hints that *"you can place an id on every single word in a phrase"*. This cell realises that
idea literally: every distinct word in the corpus is assigned a stable integer ID, so a prompt becomes a
sequence of IDs.

Two safeguards follow from it:

* the ID map is **deterministic** (tokens sorted before IDs are handed out), so it is reproducible across
  runs and machines;
* the vocabulary is fitted on the **training split only**, so no test vocabulary leaks into training.

A hashed bag-of-ids representation was also measured as a predictive feature block and **lowered** the
grouped-CV ROC-AUC (hashing collisions and the loss of IDF reweighting). The word-ID machinery is
therefore kept for inspection and for the near-duplicate safeguard, and is deliberately *not* wired into
the predictive feature union — see Cell 5 for the measured comparison.

In [ ]:
word_id_encoder = feature_builder.build_word_id_encoder(train_texts)

print(f"distinct words in the training corpus : {word_id_encoder.vocabulary_size:,}")

print("First 15 token -> ID assignments (sorted tokens, so IDs are stable):")
for token, wid in list(word_id_encoder.token_to_id_.items())[:15]:
    print(f"    {token:<18} -> {wid}")

print()
example_text = train_texts[2]
example_ids = word_id_encoder.transform([example_text])[0]
example_tokens = preprocessor.tokenize(preprocessor.normalize_text(example_text))
print(f"Example prompt: {example_text[:150]!r}")
print()
print(f"{'token':<16}{'word ID':>8}   {'token':<16}{'word ID':>8}")
for i in range(0, min(24, len(example_tokens)), 2):
    left_t, left_id = example_tokens[i], example_ids[i]
    right = example_tokens[i + 1] if i + 1 < len(example_tokens) else ""
    right_id = example_ids[i + 1] if i + 1 < len(example_ids) else ""
    print(f"{left_t:<16}{left_id:>8}   {right:<16}{right_id:>8}")

print()
print(f"tokens in example      : {len(example_tokens)}")
print(f"unseen tokens (-> -1)  : {sum(1 for i in example_ids if i == -1)}")
print(f"distinct IDs in example: {len(set(example_ids))}")

## Cell 5 — feature blocks and the word-ID contribution check

Three blocks are available:

| Block | Description |
| --- | --- |
| `word_tfidf` | word 1-2 grams, `min_df=2`, sublinear TF, `strip_accents='unicode'` |
| `char_tfidf` | `char_wb` 2-6 grams, `min_df=2`, sublinear TF |
| `word_id` | hashed bag-of-word-IDs plus ID-sequence statistics |

The `word_id` block is measured here against the same base block so the decision to exclude it from the
predictive union is backed by a number rather than an opinion.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import scipy.sparse as sp

y, label_encoder = model_trainer.encode_labels(train_df['label'])
print(f"label encoding: {list(label_encoder.classes_)} -> {np.unique(y)} (0=benign, 1=jailbreak)")
print(f"positive rate : {y.mean():.4f}")

# --- safeguard: cluster near-duplicate prompts into groups --------------
t0 = time.time()
groups = model_trainer.build_near_duplicate_groups(train_texts)
group_sizes = np.bincount(groups)
print()
print(f"near-duplicate groups : {len(group_sizes):,} distinct groups over {len(train_texts):,} rows")
print(f"largest group         : {group_sizes.max()} rows")
print(f"rows in a multi-row group: {int((group_sizes > 1).sum()):,}")
print(f"grouping took {time.time() - t0:.1f}s")

# --- measured contribution of the word-ID block --------------------------
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(np.zeros(len(y)), y, groups))

def _cv_auc(matrix, y, groups, C=32.0):
    oof = np.zeros(len(y))
    for tr, va in StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True,
                                      random_state=RANDOM_STATE).split(np.zeros(len(y)), y, groups):
        model = LogisticRegression(C=C, max_iter=3000, solver="liblinear",
                                    class_weight="balanced", random_state=RANDOM_STATE)
        model.fit(matrix[tr], y[tr])
        oof[va] = model.predict_proba(matrix[va])[:, 1]
    return float(roc_auc_score(y, oof))

char_union = feature_builder.build_feature_union(("char_tfidf",))
X_char = char_union.fit_transform(train_norm)
auc_char_only = _cv_auc(X_char, y, groups)

word_id_matrix = feature_builder.WordIdVectorizer(n_features=4096).fit_transform(train_norm)
X_char_wid = sp.hstack([X_char, word_id_matrix * 0.5], format="csr")
auc_char_wordid = _cv_auc(X_char_wid, y, groups)

print()
print(f"char_tfidf alone              : grouped-CV ROC-AUC = {auc_char_only:.5f}")
print(f"char_tfidf + word_id (w=0.5)  : grouped-CV ROC-AUC = {auc_char_wordid:.5f}")
print(f"delta from the word-ID block  : {auc_char_wordid - auc_char_only:+.5f}")
print()
print("=> the word-ID block is excluded from the predictive feature union; the word-ID")
print("   encoder is retained for inspection and for the near-duplicate safeguard.")